In [1]:
import requests
import time
from bs4 import BeautifulSoup
from datetime import date, timedelta
from pathlib import Path
from urllib import robotparser
from urllib.parse import urlparse
import json
import logging
import pandas as pd

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

TAGESSCHAU_BASE = "https://www.tagesschau.de"
API_BASE        = "https://www.tagesschau.de/api2u"
ROBOTS_URL      = f"{TAGESSCHAU_BASE}/robots.txt"
ROBOTS_USER_AGENT = "AwesomeTagesschauScraper"
OUTPUT_DIR      = Path("data/raw/tagesschau")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; AwesomeTagesschauScraper/1.0; "
        "+https://huggingface.co/datasets/stefan-it/awesome-tagesschau)"
    )
}

REQUEST_DELAY = 5

# Change: dry-run is now off for the real crawl.
DRY_RUN = False
DRY_RUN_MAX_DAYS = 2
DRY_RUN_MAX_ARTICLES = 20

# Change: run the production crawl in two windows so each half is saved separately.
RUN_WINDOWS = [
    (date(2025, 8, 1), date(2025, 10, 31)),
    (date(2025, 11, 1), date(2026, 1, 31)),
]

# Change: autosave/checkpoint settings reduce the cost of long-run failures.
AUTOSAVE_EVERY = 25
RESUME_FROM_CHECKPOINT = True

# Change: fail fast if payloads or success rate look wrong.
REQUIRED_FIELDS = ("title", "date", "detailsweb")
MIN_SUCCESS_RATE = 0.90

In [5]:
def generate_date_range(start: date, end: date):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)


In [6]:
def load_robots_parser() -> robotparser.RobotFileParser:
    response = requests.get(ROBOTS_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    if not response.text.strip():
        raise RuntimeError("robots.txt is empty; aborting crawl")

    rp = robotparser.RobotFileParser()
    rp.set_url(ROBOTS_URL)
    rp.parse(response.text.splitlines())
    return rp


ROBOTS = load_robots_parser()
crawl_delay = ROBOTS.crawl_delay(ROBOTS_USER_AGENT)
if crawl_delay is None:
    crawl_delay = ROBOTS.crawl_delay("*")
if crawl_delay:
    REQUEST_DELAY = max(REQUEST_DELAY, crawl_delay)


def _ensure_allowed(url: str) -> None:
    if not ROBOTS.can_fetch(ROBOTS_USER_AGENT, url):
        raise RuntimeError(f"robots.txt disallows crawling {url}")


def _get(url: str, retries: int = 3) -> requests.Response | None:
    _ensure_allowed(url)
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            time.sleep(REQUEST_DELAY)
            return r
        except requests.RequestException as exc:
            wait = 2 ** attempt
            log.warning("  Request failed (%s), retrying in %ds …", exc, wait)
            time.sleep(wait)
    log.error("  Giving up on %s", url)
    return None


def _extract_article_links(soup: BeautifulSoup) -> set:
    """
    Extract article links from a parsed archive page.
    """
    links = set()
    container = soup.find("div", class_="container")
    if not container:
        return links

    for a in container.find_all("a", href=True):
        href = a["href"]
        # Skip external links (e.g. DW, sportschau sub-domains)
        if not href.startswith("/"):
            continue
        # Skip non-article paths
        if any(href.startswith(p) for p in ("/archiv", "/suche", "/video", "/sendung")):
            continue
        if href.endswith(".html"):
            links.add(href)
    return links


def collect_archive_links(start: date, end: date) -> set:
    all_links = set()

    for d in generate_date_range(start, end):
        log.info("  Crawling archive for %s", d.isoformat())
        url = f"{TAGESSCHAU_BASE}/archiv?datum={d.isoformat()}"

        r = _get(url)
        if r is None:
            log.warning("  Skipping %s — request failed", d.isoformat())
            continue

        soup = BeautifulSoup(r.text, "html.parser")
        new_links = _extract_article_links(soup)

        all_links |= new_links
        log.info("    found %d links (%d total so far)", len(new_links), len(all_links))

    return all_links

In [7]:
def fetch_article(path: str):
    """
    Fetch full article JSON from the Tagesschau API2u endpoint.
    """
    url = f"{API_BASE}{path}"
    r = _get(url)
    if r is None:
        return None
    try:
        return r.json()
    except ValueError:
        log.warning("  Non-JSON response for %s", path)
        return None

In [8]:
def validate_article_payload(path: str, payload: dict) -> None:
    # Change: explicit assertions stop the run as soon as the API shape changes.
    assert isinstance(payload, dict), f"Expected dict payload for {path}"
    missing = [field for field in REQUIRED_FIELDS if field not in payload]
    assert not missing, f"Missing required fields for {path}: {missing}"


def load_checkpoint(checkpoint_file: Path, state_file: Path):
    # Change: load prior progress so a failed long crawl can resume.
    articles = []
    processed_paths = set()
    failed = []

    if checkpoint_file.exists():
        with checkpoint_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if line:
                    articles.append(json.loads(line))

    if state_file.exists():
        state = json.loads(state_file.read_text(encoding="utf-8"))
        processed_paths = set(state.get("processed_paths", []))
        failed = state.get("failed", [])

    return articles, processed_paths, failed


def autosave_progress(
    checkpoint_file: Path,
    state_file: Path,
    pending_articles: list,
    processed_paths: set,
    failed: list,
) -> None:
    # Change: write partial results during the run so progress is not only in memory.
    if pending_articles:
        with checkpoint_file.open("a", encoding="utf-8") as handle:
            for article in pending_articles:
                handle.write(json.dumps(article, ensure_ascii=False) + "\n")
        pending_articles.clear()

    state = {
        "processed_paths": sorted(processed_paths),
        "failed": failed,
    }
    state_file.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")


def run_window(start_date: date, end_date: date):
    # Change: one crawl window writes its own files so long runs are split into safer chunks.

    # Change: the dry run keeps the exact same code path but narrows the date window.
    if DRY_RUN:
        end_date = min(end_date, start_date + timedelta(days=DRY_RUN_MAX_DAYS - 1))

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_label = f"tagesschau_{start_date.isoformat()}_to_{end_date.isoformat()}"
    if DRY_RUN:
        run_label += "_dry_run"

    output_file = OUTPUT_DIR / f"{run_label}.csv"
    checkpoint_file = OUTPUT_DIR / f"{run_label}.checkpoint.jsonl"
    state_file = OUTPUT_DIR / f"{run_label}.state.json"

    articles = []
    failed   = []
    processed_paths = set()
    pending_articles = []

    if RESUME_FROM_CHECKPOINT:
        articles, processed_paths, failed = load_checkpoint(checkpoint_file, state_file)
        if articles or processed_paths or failed:
            log.info("Loaded checkpoint: %d articles, %d processed paths, %d failed", len(articles), len(processed_paths), len(failed))

    log.info("Collecting archive links")
    archive_links = sorted(collect_archive_links(start_date, end_date))
    assert archive_links, "Archive crawl returned zero article links"

    # Change: cap the article count in dry-run mode so validation stays quick.
    if DRY_RUN:
        archive_links = archive_links[:DRY_RUN_MAX_ARTICLES]

    if processed_paths:
        archive_links = [path for path in archive_links if path not in processed_paths]
    log.info("Total unique article paths found: %d", len(archive_links))

    log.info("Fetching article details from API")
    for i, path in enumerate(sorted(archive_links), 1):
        log.info("  [%d/%d] %s", i, len(archive_links), path)
        data = fetch_article(path)
        if data:
            validate_article_payload(path, data)
            articles.append(data)
            pending_articles.append(data)
            processed_paths.add(path)
        else:
            failed.append(path)

        # Change: autosave every N attempts so a long run can be resumed.
        if i % AUTOSAVE_EVERY == 0:
            autosave_progress(checkpoint_file, state_file, pending_articles, processed_paths, failed)

    autosave_progress(checkpoint_file, state_file, pending_articles, processed_paths, failed)
    log.info("Done")
    log.info("Successfully fetched: %d", len(articles))
    log.info("Failed: %d", len(failed))
    if failed:
        log.info("Failed paths: %s", json.dumps(failed, ensure_ascii=False))

    attempts = len(articles) + len(failed)
    assert attempts > 0, "No article fetches were attempted"
    success_rate = len(articles) / attempts
    assert success_rate >= MIN_SUCCESS_RATE, (
        f"Success rate {success_rate:.1%} below threshold {MIN_SUCCESS_RATE:.1%}"
    )

    df = pd.json_normalize(articles)
    assert not df.empty, "Normalized dataframe is empty"
    df.to_csv(output_file, index=False)
    log.info("Saved to %s", output_file)
    return df, output_file

In [20]:
# Change: run the first production batch explicitly in this cell only.
FIRST_WINDOW = RUN_WINDOWS[0]
log.info("Starting first batch %s to %s", FIRST_WINDOW[0].isoformat(), FIRST_WINDOW[1].isoformat())
df_first, output_file_first = run_window(*FIRST_WINDOW)
print(output_file_first)
print(df_first.shape)

11:26:24  INFO  Starting first batch 2025-08-01 to 2025-10-31
11:26:24  INFO  Collecting archive links
11:26:24  INFO    Crawling archive for 2025-08-01
11:26:35  INFO      found 37 links (37 total so far)
11:26:35  INFO    Crawling archive for 2025-08-02
11:26:44  INFO      found 26 links (63 total so far)
11:26:44  INFO    Crawling archive for 2025-08-03
11:26:52  INFO      found 23 links (86 total so far)
11:26:52  INFO    Crawling archive for 2025-08-04
11:27:03  INFO      found 35 links (121 total so far)
11:27:03  INFO    Crawling archive for 2025-08-05
11:27:13  INFO      found 42 links (163 total so far)
11:27:13  INFO    Crawling archive for 2025-08-06
11:27:24  INFO      found 38 links (201 total so far)
11:27:24  INFO    Crawling archive for 2025-08-07
11:27:36  INFO      found 50 links (251 total so far)
11:27:36  INFO    Crawling archive for 2025-08-08
11:27:46  INFO      found 39 links (290 total so far)
11:27:46  INFO    Crawling archive for 2025-08-09
11:27:55  INFO    

data/raw/tagesschau/tagesschau_2025-08-01_to_2025-10-31.csv
(3244, 42)


In [9]:
# Change: run the second production batch later, only when you decide to execute this cell.
SECOND_WINDOW = RUN_WINDOWS[1]
log.info("Starting second batch %s to %s", SECOND_WINDOW[0].isoformat(), SECOND_WINDOW[1].isoformat())
df_second, output_file_second = run_window(*SECOND_WINDOW)
print(output_file_second)
print(df_second.shape)

00:22:29  INFO  Starting second batch 2025-11-01 to 2026-01-31
00:22:29  INFO  Collecting archive links
00:22:29  INFO    Crawling archive for 2025-11-01
00:22:37  INFO      found 20 links (20 total so far)
00:22:37  INFO    Crawling archive for 2025-11-02
00:22:45  INFO      found 20 links (40 total so far)
00:22:45  INFO    Crawling archive for 2025-11-03
00:22:55  INFO      found 35 links (75 total so far)
00:22:55  INFO    Crawling archive for 2025-11-04
00:23:05  INFO      found 44 links (119 total so far)
00:23:05  INFO    Crawling archive for 2025-11-05
00:23:15  INFO      found 45 links (164 total so far)
00:23:15  INFO    Crawling archive for 2025-11-06
00:23:25  INFO      found 46 links (210 total so far)
00:23:25  INFO    Crawling archive for 2025-11-07
00:23:36  INFO      found 35 links (245 total so far)
00:23:36  INFO    Crawling archive for 2025-11-08
00:23:44  INFO      found 26 links (271 total so far)
00:23:44  INFO    Crawling archive for 2025-11-09
00:23:52  INFO   

data/raw/tagesschau/tagesschau_2025-11-01_to_2026-01-31.csv
(3076, 42)
